# 悬臂梁问题

**类别：** 仿真

来源：[https://www.hexaly.com/templates/cantilevered-beam-problem](https://www.hexaly.com/templates/cantilevered-beam-problem)

## 问题描述

**悬臂梁问题** 由设计 I 型梁的横截面组成,以使梁在一定应力下不会变形或破坏的情况下获得最小体积。有关更多详细信息,请参阅 [Wikipedia](https://en.wikipedia.org/wiki/Cantilever)。我们记 L、H、h1、b1 和 b2 为梁的尺寸。它们根据下述图片来定义。

梁的体积由下式给出:

$$V = f(H, h_1, b_1, b_2) = \left[2h_1b_1 + \left(H - 2h_1\right)b_2\right]L$$

约束条件为:

- 梁根部处的最大弯曲应力,定义为:

  $$g_1(H, h_1, b_1, b_2) = \frac{P L H}{2I}$$

- 梁尖端处的最大挠度,定义为:

  $$g_2(H, h_1, b_1, b_2) = \frac{P L^3}{3EI}$$

	

### 学习要点

- 创建一个[外部函数](https://www.hexaly.com/docs/last/mathematicaloperators/externalfunctions.html),返回多个值
- 在外部函数上启用[代理建模](https://www.hexaly.com/docs/last/mathematicaloperators/externalfunctions.html#surrogate-modeling)
- 为该函数设置求值次数限制

注意:本示例旨在说明如何定义一个返回数组的外部函数,以及如何在简单问题上使用代理建模。由于本问题中使用的外部函数计算开销很小,因此不使用代理建模功能也可以求解。事实上,代理建模仅在目标函数的计算代价昂贵时才有用。在本示例中,我们仅出于演示目的使用它。

## 建模方法

悬臂梁问题的 Hexaly 模型使用整型和浮点型决策变量。我们首先声明三个浮点型决策变量 H、b1 和 b2。这些变量的定义域分别为 [3.0, 7.0]、[2.0, 12.0] 和 [3.0, 7.0]。最后一个决策变量 h1 是离散的。它被声明为一个整数,表示包含可能取值数组中的索引,这些取值为 {0.1, 0.26, 0.35, 0.5, 0.65, 0.75, 0.9, 1.0}。因此该变量的定义域为 [0, 7]。

梁的体积、弯曲应力和挠度通过一个[外部函数](https://www.hexaly.com/docs/last/mathematicaloperators/externalfunctions.html)计算。它通过 HxExternalArgumentValues 对象接收参数值,并返回一个包含这些梁尺寸对应三个函数值的数组。为了计算由外部函数返回的值,我们创建一个调用表达式。我们使用前两个返回值(弯曲应力和挠度)来约束模型。第三个返回值(体积)就是要最小化的目标函数。

为了使用[代理建模特性](https://www.hexaly.com/docs/last/mathematicaloperators/externalfunctions.html#external-functions-surrogate-modeling),我们调用该函数的 HxExternalContext 上可用的 enableSurrogateModeling 方法。该方法返回 HxSurrogateParameters,可以在其上设置对该函数的最大调用次数。由于该函数在实际中通常计算代价昂贵,因此将搜索限制在合理的时间内非常有用。

## Python 实现

In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys

# Constant declaration
P = 1000
E = 10.0e6
L = 36
possibleValues = [0.1, 0.26, 0.35, 0.5, 0.65, 0.75, 0.9, 1.0]

# External function
def evaluate(arguments_values):
    # Argument retrieval
    fH = arguments_values[0]
    fh1 = possibleValues[arguments_values[1]]
    fb1 = arguments_values[2]
    fb2 = arguments_values[3]

    # Big computation
    I = 1.0 / 12.0 * fb2 * pow(fH - 2 * fh1, 3) + 2 * (1.0 / 12.0 * fb1
        * pow(fh1, 3) + fb1 * fh1 * pow(fH - fh1, 2) / 4.0)

    # Constraint computations
    g1 = P * L * fH / (2 * I)
    g2 = P * L**3 / (3 * E * I)

    # Objective function computation
    f = (2 * fh1 * fb1 + (fH - 2 * fh1) * fb2) * L

    return g1, g2, f

def main(evaluation_limit, output_file):
    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        # Declare the optimization model
        model = optimizer.model

        # Numerical decisions
        H = model.float(3.0, 7.0)
        h1 = model.int(0, 7)
        b1 = model.float(2.0, 12.0)
        b2 = model.float(0.1, 2.0)

        # Create and call the external function
        func = model.create_double_array_external_function(evaluate)
        func_call = model.call(func)
        # Add the operands
        func_call.add_operand(H)
        func_call.add_operand(h1)
        func_call.add_operand(b1)
        func_call.add_operand(b2)

        # Enable surrogate modeling
        surrogate_params = func.external_context.enable_surrogate_modeling()

        # Constraint on bending stress
        model.constraint(func_call[0] <= 5000)
        # Constraint on deflection at the tip of the beam
        model.constraint(func_call[1] <= 0.10)

        objective = func_call[2]
        model.minimize(objective)
        model.close()

        # Parameterize the optimizer
        surrogate_params.evaluation_limit = evaluation_limit

        optimizer.solve()

        # Write the solution in a file with the following format:
        # - The value of the minimum found
        # - The location (H; h1; b1; b2) of the minimum
        if output_file is not None:
            with open(output_file, 'w') as f:
                f.write("Objective value: " + str(objective.value) + "\n")
                f.write("Point (H;h1;b1;b2): (" + str(H.value) + ";"
                    + str(possibleValues[h1.value]) + ";" + str(b1.value) + ";"
                    + str(b2.value) + ")")


if __name__ == '__main__':
    output_file = sys.argv[1] if len(sys.argv) > 1 else None
    evaluation_limit = int(sys.argv[2]) if len(sys.argv) > 2 else 30

    main(evaluation_limit, output_file)